## What This Notebook Does

Exploratory Data Analysis (EDA) is the process of **visually and statistically
understanding the data before building any model**.

In this notebook we answer real Formula 1 questions:
- Who are the most dominant drivers of the 2018–2026 era?
- Which constructors dominated which seasons?
- Which circuits produce the most DNFs?
- Are there real circuit specialists in the data?
- Which engineered features correlate most strongly with race outcomes?

# Import Libs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visual Style

In [ ]:
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'DejaVu Sans'
})

# colour palette
F1_RED = '#E8002D' #brand
F1_DARK = '#15151E' #background
F1_SILVER = '#C0C0C0' #silver

HEATMAP_CMAP = 'RdYlGn_r' #red - dng, green - good

# Load Data

In [ ]:
df = pd.read_csv('../data/processed/f1_features_2018_2026.csv')

print(f'  Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'  Seasons: {df.season.min()} — {df.season.max()}')
print(f'  Unique drivers:      {df.driver_id.nunique()}')
print(f'  Unique constructors: {df.constructor_id.nunique()}')
print(f'  Unique circuits:     {df.circuit_id.nunique()}')
print(f'  Total race entries:  {len(df):,}')


# Overview

In [ ]:
df.info()

In [ ]:
df[['position', 'grid', 'points', 'dnf', 'podium']].describe().round(2)

# Missing values

In [ ]:
key_cols = ['season', 'round', 'circuit_id', 'driver_id', 'constructor_id',
            'grid', 'position', 'points', 'dnf', 'podium']

nan_count = df[key_cols].isnull().sum()

In [ ]:
nan_count[nan_count > 0] if nan_count.sum() > 0 else 'None -dataset clean'

# Points Distribution

In [ ]:
f'Race with 0 points: {(df.points == 0).sum():,} ({(df.points == 0).mean()*100:.1f}%)'

In [ ]:
f'Races with 1+ points: {(df.points > 0).sum():,} ({(df.points>0).mean()*100:.1f}%)'

In [ ]:
f'DNF rate oveall: {df.dnf.mean()*100:.1f}%'

In [ ]:
f'Podium rate: {df.podium.mean()*100:.1f}%'

# SECTION 1 — Driver Analysis

### wins

In [ ]:
wins_by_driver = (
    df[df.position == 1]
    .groupby('driver_name')
    .size()
    .reset_index(name='wins')
    .sort_values('wins', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(13, 7))

bars = ax.barh(
    wins_by_driver['driver_name'],
    wins_by_driver['wins'],
    color=F1_RED,
    edgecolor='white',
    linewidth=0.5,
    height=0.7
)

# Add value labels to end of each bar
for bar, val in zip(bars, wins_by_driver['wins']):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height() / 2,
        str(int(val)),
        va='center', ha='left', fontsize=11, fontweight='bold'
    )

ax.set_xlabel('Race Wins', fontsize=13)
ax.set_title('Top 15 Drivers by Race Wins — 2018 to 2026', fontsize=15, fontweight='bold')
ax.set_xlim(0, wins_by_driver['wins'].max() + 8)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/04_wins_by_driver.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
print(f"  Most wins: {wins_by_driver.iloc[0]['driver_name']} "
      f"({wins_by_driver.iloc[0]['wins']} wins)")
print(f"  Top 3 drivers account for "
      f"{wins_by_driver.head(3)['wins'].sum()} of "
      f"{(df.position == 1).sum()} total wins "
      f"({wins_by_driver.head(3)['wins'].sum() / (df.position==1).sum() * 100:.1f}%)")

# Podiums

In [ ]:
podiums_by_driver = (
    df[df.podium == 1]
    .groupby('driver_name')
    .size()
    .reset_index(name='podiums')
    .sort_values('podiums', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(13,7))

colors = [F1_RED if i == 0 else '#4A90D9' for i in range(len(podiums_by_driver))]

bars = ax.barh(
    podiums_by_driver['driver_name'],
    podiums_by_driver['podiums'],
    color = colors,
    edgecolor = 'white',
    linewidth = 0.5,
    height=0.7
)

for bar, val in zip(bars, podiums_by_driver['podiums']):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        str(int(val)),
        va = 'center', ha='left', fontsize=11, fontweight='bold'
    )

ax.set_xlabel('Podium Finishes (P1 + P2 + P3)', fontsize=13)
ax.set_title('Top 15 Drivers by Podiums — 2018 to 2026', fontsize=15, fontweight='bold')
ax.set_xlim(0, podiums_by_driver['podiums'].max() + 12)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/04_podiums_by_driver.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

print('KEY INSIGHT:')
top = podiums_by_driver.iloc[0]
print(f"  Most podiums: {top['driver_name']} ({top['podiums']} podiums)")

# Points

In [ ]:
points_by_driver = (
    df.groupby('driver_name')['points']
    .sum()
    .reset_index(name='total_points')
    .sort_values('total_points', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(13,7))

palette = sns.color_palette('Blues_d', n_colors=len(points_by_driver))[::-1]

bars = ax.barh(
    points_by_driver['driver_name'],
    points_by_driver['total_points'],
    color=palette,
    edgecolor='white',
    linewidth=0.4,
    height=0.7
)

for bar, val in zip(bars, points_by_driver['total_points']):
    ax.text(
        bar.get_width() +15,
        bar.get_y() + bar.get_height() / 2,
        f'{int(val):,}',
        va='center', ha='left', fontsize=10, fontweight='bold'
    )

ax.set_xlabel('Total Championship Points', fontsize=13)
ax.set_title('Top 15 Drivers by Total Points 2018-2026', fontsize=15, fontweight='bold')
ax.set_xlim(0, points_by_driver['total_points'].max() +200)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/04_points_by_driver.png', dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
top=points_by_driver.iloc[0]
print(f"Most points: {top['driver_name']} ({top['total_points']:,.0f} pts)")

# DNF Rate

In [ ]:
driver_stats = (
    df.groupby('driver_name')
    .agg(
        starts = ('position', 'count'),
        dnfs = ('dnf', 'sum')
    )
    .reset_index()
)

driver_stats['dnf_rate'] = driver_stats['dnfs'] / driver_stats['starts']

In [ ]:
driver_dnf = (
    driver_stats[driver_stats.starts >=20]
    .sort_values('dnf_rate', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(13,7))

bar_colors = [
    F1_RED if r > 0.15 else '#F5A623' if r>0.08 else '27AE60'
    for r in driver_dnf['dnf_rate']
]

bars = ax.barh(
    driver_dnf['driver_name'],
    driver_dnf['dnf_rate'] *100,
    color=bar_colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.7
)

for bar, val, starts in zip(bars, driver_dnf['dnf_rate'], driver_dnf['starts']):
    ax.text(
        bar.get_width() *0.2,
        bar.get_y() + bar.get_height() /2,
        f'{val*100:.1f}% ({int(starts)} starts)',
        va='center', ha='left',fontsize=9
    )

ax.set_xlabel('DNF Rate (%)', fontsize=13)
ax.set_title('Driver DNF Rate 2018-2026 (min. 20 starts)', fontsize=15, fontweight='bold')
ax.set_xlim(0, driver_dnf['dnf_rate'].max() * 100+8)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_el = [
    Patch(facecolor=F1_RED, label='High risk > 15%'),
    Patch(facecolor='#F5A623', label='Medium risk 8-15%'),
    Patch(facecolor='#27AE60', label='Low risk <8%')
]
ax.legend(handles=legend_el, loc='lower right', fontsize=9)


plt.tight_layout()
plt.savefig('../output/04_dnf_rate_by_driver.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
most_reliable = driver_stats[driver_stats.starts >=20].sort_values('dnf_rate').iloc[0]
least_reliable = driver_stats[driver_stats.starts >= 20].sort_values('dnf_rate').iloc[-1]
print(f"Most reliable: {most_reliable['driver_name']}"
      f"({most_reliable['dnf_rate']*100:.1f}% DNF rate)")
print(f"Most retirements: {least_reliable['driver_name']}"
      f"({least_reliable['dnf_rate']*100:.1f}% DNF rate)")

# Section 2 Constructor Analysis

### wins

In [ ]:
wins_by_constructor = (
    df[df.position == 1]
    .groupby('constructor_id')
    .size()
    .reset_index(name='wins')
    .sort_values('wins', ascending=False)
)

team_colors = {
    'mercedes':     '#00D2BE',
    'red_bull':     '#3671C6',
    'ferrari':      '#DC0000',
    'mclaren':      '#FF8000',
    'alpine':       '#0090FF',
    'aston_martin': '#006F62',
    'williams':     '#64C4FF',
    'alphatauri':   '#2B4562',
    'toro_rosso':   '#527eb0',
    'sauber':   '#27ff60',
    'alfa':         '#900000',
    'haas':         '#B6BABD',
    'renault':      '#FFF500',
    'racing_point': '#F596C8',
}

bad_colors = [
    team_colors.get(cid, '#888888')
    for cid in wins_by_constructor['constructor_id']
]

fig, ax = plt.subplots(figsize=(13,6))

bars = ax.bar(
    wins_by_constructor['constructor_id'],
    wins_by_constructor['wins'],
    color=bad_colors,
    edgecolor='white',
    linewidth=0.6,
    width=0.7
)

for bar, val in zip(bars, wins_by_constructor['wins']):
    ax.text(
        bar.get_x() + bar.get_width() /2,
        bar.get_height() + 0.8,
        str(int(val)),
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_ylabel('Race Wins', fontsize=13)
ax.set_xlabel('Constructor', fontsize=13)
ax.set_title('Race Wins by Constructor 2018-2026', fontsize=15, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/04_wins_by_constructor.png', dpi = 150, bbox_inches = 'tight')
plt.show()



In [ ]:
print('KEY INSIGHT')
top = wins_by_constructor.iloc[0]
total = wins_by_constructor['wins'].sum()
print(f"Most wins: {top["constructor_id"]} ({top['wins']} wins, " 
      f"{top['wins']/total*100:.1f}% of all races)")
print(f"Top 2 constructors share"
      f"{wins_by_constructor.head(2)['wins'].sum()} of {total} total wins"
      f"({wins_by_constructor.head(2)['wins'].sum()/total*100:.1f}%)")

### points

In [ ]:
constructor_season_points = (
    df.groupby(['season', 'constructor_id'])['points']
    .sum()
    .reset_index()
    .pivot(index='constructor_id', columns='season', values='points')
    .fillna(0)
)

In [ ]:
constructor_season_points['total'] = constructor_season_points.sum(axis=1)
constructor_season_points =(
    constructor_season_points
    .sort_values('total', ascending=False)
    .drop(columns = 'total')
)

In [ ]:
fig, ax = plt.subplots(figsize=(14,8))

sns.heatmap(
    constructor_season_points,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Total Points'},
    ax=ax,
    annot_kws={'size': 9}
)


ax.set_title(
    'Constructor Championship Points by Season 2018-2026',
    fontsize=15, fontweight='bold', pad=15
)

ax.set_xlabel('Season', fontsize=13)
ax.set_ylabel('Constructor', fontsize=13)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y',rotation=0)

plt.tight_layout()
plt.savefig('../output/04_constructor_points_heatmap.png', dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
best_season = constructor_season_points.max(axis=1).idxmax()
best_pts    = constructor_season_points.max(axis=1).max()
best_yr     = constructor_season_points.loc[best_season].idxmax()
print(f"  Single-season record: {best_season} in {best_yr} ({best_pts:.0f} pts)")
print("  The heatmap shows clear dominant bands — validating that")
print("  constructor_points_current_season is a meaningful feature.")

# Constructor Reliability
DNF Rate

In [ ]:
constructor_reliability = (
    df.groupby('constructor_id')
    .agg(
        starts = ('position', 'count'),
        dnfs = ('dnf', 'sum')
    )
    .reset_index()
)

In [ ]:
constructor_reliability['dnf_rate'] = (
    constructor_reliability['dnfs'] / constructor_reliability['starts']
)

constructor_reliability['finished_rate'] = 1 - constructor_reliability['dnf_rate']
constructor_reliability = constructor_reliability.sort_values('dnf_rate')

In [ ]:
fig, ax = plt.subplots(figsize=(13,7))

bar_colors = [
    '#27AE60' if r< 0.08 else '#F5A623' if r<0.15 else F1_RED
    for r in constructor_reliability['dnf_rate']
]

bars = ax.barh(
    constructor_reliability['constructor_id'],
    constructor_reliability['dnf_rate']*100,
    color = bar_colors,
    edgecolor ='white',
    linewidth=0.5,
    height=0.7
)

for bar, rate, starts in zip(
    bars,
    constructor_reliability['dnf_rate'],
    constructor_reliability['starts']
):
    ax.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f'{rate*100:.1f}%  ({int(starts)} starts)',
        va='center', ha='left', fontsize=9
    )

ax.set_xlabel('DNF Rate (%)', fontsize=13)
ax.set_title('Constructor DNF Rate 2018-2026', fontsize=15,fontweight='bold')
ax.set_xlim(0, constructor_reliability['dnf_rate'].max() *100 +8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_el = [
    Patch(facecolor='#27AE60', label='Reliable <8%'),
    Patch(facecolor='#F5A623', label='Average 8-15%'),
    Patch(facecolor=F1_RED, label='Unreliable >15%'),
]
ax.legend(handles=legend_el, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../output/04_constructor_reliability.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
print('KEY INSIGHT:')
most_reliable = constructor_reliability.iloc[0]
least_reliable = constructor_reliability.iloc[-1]
print(f"Most reliable: {most_reliable['constructor_id']}"
      f"({most_reliable['dnf_rate']*100:.1f}% DNF rate)")
print(f"Least reliable: {least_reliable['constructor_id']}"
      f"({least_reliable['dnf_rate']*100:.1f}% DNF rate)")

# Section 3 - Season Trends

wins

In [ ]:

wins_season = (
    df[df.position == 1]
    .groupby(['season', 'constructor_id'])
    .size()
    .reset_index(name='wins')
)

wins_pivot = (
    wins_season
    .pivot(index='season', columns='constructor_id', values='wins')
    .fillna(0)
)

# Sort columns by total wins so dominant teams appear first in stack
col_order = wins_pivot.sum().sort_values(ascending=False).index
wins_pivot = wins_pivot[col_order]

fig, ax = plt.subplots(figsize=(14, 7))

colors_map = {
    'mercedes':     '#00D2BE',
    'red_bull':     '#3671C6',
    'ferrari':      '#DC0000',
    'mclaren':      '#FF8000',
    'alpine':       '#0090FF',
    'aston_martin': '#006F62',
    'williams':     '#64C4FF',
    'alphatauri':   '#2B4562',
    'alfa':         '#900000',
    'haas':         '#B6BABD',
    'renault':      '#FFF500',
    'racing_point': '#F596C8',
}

bottom = np.zeros(len(wins_pivot))
for col in wins_pivot.columns:
    color = colors_map.get(col, '#AAAAAA')
    bars  = ax.bar(
        wins_pivot.index,
        wins_pivot[col],
        bottom=bottom,
        label=col,
        color=color,
        edgecolor='white',
        linewidth=0.5,
        width=0.7
    )
    bottom += wins_pivot[col].values

# Annotate total wins per season at top of bar
for i, season in enumerate(wins_pivot.index):
    total = wins_pivot.loc[season].sum()
    ax.text(
        season, total + 0.4,
        str(int(total)),
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_ylabel('Race Wins', fontsize=13)
ax.set_xlabel('Season', fontsize=13)
ax.set_title('Race Wins per Season by Constructor — 2018 to 2026',
             fontsize=15, fontweight='bold')
ax.legend(loc='upper left', fontsize=8, ncol=2, bbox_to_anchor=(1.01, 1))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/04_wins_per_season.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
unique_winners = (
    df[df.position == 1]
    .groupby('season')['driver_id']
    .nunique()
    .reset_index()
    .rename(columns={'driver_id': 'unique_winners'})
)
most_competitive = unique_winners.loc[unique_winners.unique_winners.idxmax()]
least_competitive = unique_winners.loc[unique_winners.unique_winners.idxmin()]
print(f"  Most competitive season: {int(most_competitive.season)} "
      f"({int(most_competitive.unique_winners)} different race winners)")
print(f"  Most dominant season:    {int(least_competitive.season)} "
      f"(only {int(least_competitive.unique_winners)} different race winners)")


points trends

In [ ]:
top_drivers = (
    df.groupby('driver_name')['points']
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
    .tolist()
)

season_points = (
    df[df.driver_name.isin(top_drivers)]
    .groupby(['season', 'driver_name'])['points']
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 7))

palette = sns.color_palette('tab10', n_colors=8)  # ← tab10

for i, driver in enumerate(top_drivers):
    data = season_points[season_points.driver_name == driver]
    ax.plot(
        data['season'],
        data['points'],
        marker='o',
        linewidth=2.5,
        markersize=7,
        label=driver,
        color=palette[i]
    )

    if not data.empty:
        last = data.iloc[-1]
        ax.annotate(
            driver.split()[-1],
            xy=(last.season, last.points),
            xytext=(4, 0),
            textcoords='offset points',
            fontsize=8,
            color=palette[i]
        )

ax.set_xlabel('Season', fontsize=13)
ax.set_ylabel('Championship Points', fontsize=13)
ax.set_title('Season Points — Top 8 Drivers — 2018 to 2026',
             fontsize=15, fontweight='bold')
ax.set_xticks(sorted(df.season.unique()))
ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1.01, 1))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig('../output/04_points_trend_top_drivers.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
print('KEY INSIGHT:')
print("  Drivers with consistently high points across multiple seasons are")
print("  exactly who our model should predict to finish near the front.")
print("  Their dominance is captured by driver_points_current_season and")
print("  driver_avg_finish_last_3_races working together.")

dnf rate

In [ ]:
dnf_by_season =(
    df.groupby('season')
    .agg(
        total_starts=('position', 'count'),
        total_dnfs = ('dnf', 'sum')
    )
    .reset_index()
)

dnf_by_season['dnf_rate'] = dnf_by_season['total_dnfs'] / dnf_by_season['total_starts']

fig, ax = plt.subplots(figsize=(12,6))

bar_colors = [
    F1_RED if r > 0.15 else '#F5A623' if r >0.10 else '#27AE60'
    for r in dnf_by_season['dnf_rate']
]

bars = ax.bar(
    dnf_by_season['season'],
    dnf_by_season['dnf_rate'] *100,
    color=bar_colors,
    edgecolor ='white',
    linewidth=0.6,
    width=0.65,
)

for bar, val, dnfs, starts in zip(
    bars,
    dnf_by_season['dnf_rate'],
    dnf_by_season['total_dnfs'],
    dnf_by_season['total_starts'],
):
    ax.text(
        bar.get_x() + bar.get_width() /2,
        bar.get_height() +0.2,
        f'{val*100:.1f}%\n({int(dnfs)}/{int(starts)})',
        ha='center', va='bottom', fontsize=9
    )


overall_dnf = df.dnf.mean() *100
ax.axhline(
    overall_dnf,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label=f'Overall average ({overall_dnf:.1f}%)'
)

ax.set_ylabel('DNF Rate (%)', fontsize=13)
ax.set_xlabel('Season', fontsize=13)
ax.set_title('Overall DNF Rate by Season 2018-2026',
             fontsize=15, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, dnf_by_season['dnf_rate'].max()*100+5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/04_dnf_rate_by_season.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
worst  = dnf_by_season.loc[dnf_by_season.dnf_rate.idxmax()]
best   = dnf_by_season.loc[dnf_by_season.dnf_rate.idxmin()]
print(f"  Highest DNF season: {int(worst.season)} ({worst.dnf_rate*100:.1f}%)")
print(f"  Lowest DNF season:  {int(best.season)} ({best.dnf_rate*100:.1f}%)")
print(f"  Overall average:    {df.dnf.mean()*100:.1f}%")

# Section 4 Circuit Analysis

dnf

In [ ]:
circuit_dnf = (
    df.groupby('circuit_id')
    .agg(
        starts = ('position', 'count'),
        dnfs = ('dnf', 'sum')
    )
    .reset_index()
)

circuit_dnf['dnf_rate'] = circuit_dnf['dnfs'] / circuit_dnf['starts']
circuit_dnf = circuit_dnf.sort_values('dnf_rate', ascending=False)

fig, ax = plt.subplots(figsize=(14,9))

bar_colors = [
    F1_RED if r > 0.18 else
    '#F5A623' if r > 0.12 else
    '#27AE60'
    for r in circuit_dnf['dnf_rate']
]

bars = ax.barh(
    circuit_dnf['circuit_id'],
    circuit_dnf['dnf_rate'] *100,
    color=bar_colors,
    edgecolor='white',
    linewidth = 0.5,
    height=0.75
)

for bar,rate, dnfs, starts in zip(
    bars,
    circuit_dnf['dnf_rate'],
    circuit_dnf['dnfs'],
    circuit_dnf['starts']
):
    ax.text(
        bar.get_width()+0.2,
        bar.get_y() + bar.get_height() /2,
        f'{rate*100:.1f}% ({int(dnfs)} DNFs / {int(starts)} starts)',
        va = 'center', ha='left', fontsize=8.5
    )


#overall avg line
overall = df.dnf.mean() *100
ax.axvline(
    overall,
    color='black',
    linestyle='--',
    linewidth =1.5,
    label=f'Overall avg ({overall:.1f}%)'
)

ax.set_xlabel('DNF Rate {%}', fontsize=13)
ax.set_title('DNF Rate by circuit 2018-2026', fontsize=15, fontweight='bold')
ax.set_xlim(0, circuit_dnf['dnf_rate'].max() * 100 +14)
ax.invert_yaxis()
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_el = [
    Patch(facecolor=F1_RED, label='High DNF rate > 18%'),
    Patch(facecolor='#F5A623', label='Medium DNF rate 12-18%'),
    Patch(facecolor='#27AE60', label='Low DNF rate < 12%')
]

ax.legend(handles=legend_el, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../output/04_dnf_rate_by_circuit.png', dpi = 150, bbox_inches = 'tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
highest = circuit_dnf.iloc[0]
lowest = circuit_dnf.iloc[-1]
print(f"Highst DNF circuit: {highest['circuit_id']}"
      f"({highest['dnf_rate']*100:.1f}%)")
print(f"Lowest  DNF circuit: {lowest['circuit_id']}"
      f"({lowest['dnf_rate']*100:.1f}%)")
print(f"This confirms that driver_dnf_on_circuit adds value")
print(f"Beyond the global driver_dnf_rate feature")

avg finish + variance

In [ ]:
circuit_finish = (
    df.groupby('circuit_id')['position']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_finish', 'std': 'std_finish', 'count': 'starts'})
    .sort_values('std_finish', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16,8))

#left avg finish position
ax1 = axes[0]
bars1 = ax1.barh(
    circuit_finish['circuit_id'],
    circuit_finish['avg_finish'],
    color='#4A90D9',
    edgecolor='white',
    linewidth=0.4,
    height=0.75
)
ax1.axvline(
    10.5, color='black', linestyle='--',
    linewidth=1.5, label='Theoretical avg (10.5)'
)
ax1.set_xlabel('Average Finishing Position', fontsize=11)
ax1.set_title('Mean Finish Position\nby Circuit', fontsize=13, fontweight='bold')
ax1.legend(fontsize=8)
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

#right standard devitation high - sp effects
ax2 = axes[1]
colors2 = [
    F1_RED if s > 6.5 else
    '#F5A623' if s > 5.5 else
    '#4A90D9'
    for s in circuit_finish['std_finish']
]
bars2 = ax2.barh(
    circuit_finish['circuit_id'],
    circuit_finish['std_finish'],
    color=colors2,
    edgecolor='white',
    linewidth=0.4,
    height=0.75
)

for bar, val in zip(bars2, circuit_finish['std_finish']):
    ax2.text(
        bar.get_width() +0.05,
        bar.get_y() + bar.get_height() /2,
        f'{val:.2f}',
        va='center', fontsize=8
    )

ax2.set_xlabel('Std Dev of finish Position', fontsize=11)
ax2.set_title('Finishing Position Variance\nby Circuit (high = more sp effects)',
              fontsize=13, fontweight='bold')
ax2.invert_yaxis()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

ax2.set_xlabel('Std Dev of Finishing Position', fontsize=11)
ax.set_title('Finishing Position Variance\nby Circuit (high - more sp effect)', 
             fontsize=13, fontweight='bold')
ax2.invert_yaxis()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Circuit Finishing Position Analysis 2018-2026',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../output/04_circuit_finish_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT:')
highest_var = circuit_finish.iloc[0]
lowest_var  = circuit_finish.iloc[-1]
print(f"  Highest variance: {highest_var['circuit_id']} (std={highest_var['std_finish']:.2f})")
print(f"  → Specialist effects most pronounced here.")
print(f"  Lowest variance:  {lowest_var['circuit_id']} (std={lowest_var['std_finish']:.2f})")
print(f"  → Car performance dominates here more than driver specialization.")

circuit sp heatmap, top driver

In [ ]:
top12_drivers = (
    df.groupby('driver_name')['points']
    .sum()
    .sort_values(ascending=False)
    .head(12)
    .index
    .tolist()
)

# Average finishing position per driver per circuit
# Using only completed races (non-DNF) for a fair comparison
specialist_data = (
    df[
        (df.driver_name.isin(top12_drivers)) &
        (df.dnf == 0)
    ]
    .groupby(['circuit_id', 'driver_name'])['position']
    .agg(['mean', 'count'])
    .reset_index()
)

specialist_data = specialist_data[specialist_data['count'] >=2]

heatmap_data = specialist_data.pivot(
    index='circuit_id',
    columns='driver_name',
    values='mean'
)

heatmap_data['variance'] = heatmap_data.var(axis=1)
heatmap_data = heatmap_data.sort_values('variance', ascending=False)
heatmap_data = heatmap_data.drop(columns='variance')

fig, ax = plt.subplots(figsize=(16,10))

mask = heatmap_data.isna()

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn_r',
    vmin=1,
    vmax=15,
    linewidths=0.5,
    linecolor='white',
    mask=mask,
    cbar_kws={'label': 'Avg Finishing Position (lower-better)', 'shrink': 0.8},
    ax=ax,
    annot_kws={'size': 8}
)

ax.set_title(
    'Circuit Specialist Heatmap — Avg Finish Position per Driver per Circuit\n'
    '(Green = strong, Red = weak, Grey = insufficient data)',
    fontsize=14, fontweight='bold', pad=15
)

ax.set_xlabel('Driver', fontsize=12)
ax.set_ylabel('Circuit', fontsize=12)
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('../output/04_circuit_specialist_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print('KEY INSIGHT:')
print("  Clear green/red patterns confirm that circuit specialists are real.")
print("  Some drivers are consistently green at certain circuits (specialists)")
print("  and consistently red at others (weak circuits).")
print("  This directly validates driver_avg_finish_on_circuit and")
print("  driver_track_strength_score as meaningful predictive features.")


points per circuit

In [ ]:
points_specialist = (
    df[df.driver_name.isin(top12_drivers)]
    .groupby(['circuit_id', 'driver_name'])
    .agg(avg_points=('points', 'mean'), count=('points', 'count'))
    .reset_index()
)

points_specialist = points_specialist[points_specialist['count'] >= 2]

points_heatmap = points_specialist.pivot(
    index='circuit_id',
    columns='driver_name',
    values='avg_points'
)

common_circuits = [c for c in heatmap_data.index if c in points_heatmap.index]

points_heatmap = points_heatmap.loc[common_circuits]

fig, ax = plt.subplots(figsize=(16,10))

mask_pts = points_heatmap.isna()

sns.heatmap(
    points_heatmap,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    vmin=0,
    vmax=18,
    linewidths=0.5,
    linecolor='white',
    mask=mask_pts,
    cbar_kws={'label': 'Avg Points Scored (high - better)', 'shrink': 0.8},
    ax=ax,
    annot_kws={'size':8}
)

ax.set_title(
    'Avg Points Scored per Driver per Circuit\n'
    '(Green = high scoring, Red = low scoring, Grey = insufficient data)',
    fontsize=14, fontweight='bold', pad=15
)

ax.set_xlabel('Driver', fontsize=12)
ax.set_ylabel('Circuit', fontsize=12)
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('../output/04_points_per_circuit_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
print('KEY INSIGHT:')
print("  Compare this heatmap to the previous one (avg finish position).")
print("  Discrepancies between the two (e.g., a driver has a good avg finish")
print("  but low avg points) reveal circuits where that driver often DNFs")
print("  when competitive — exactly what driver_dnf_on_circuit captures.")

qual vs race delta

In [ ]:
df['race_delta'] = df['grid'] - df['position']

delta_by_circuit = (
    df[df.dnf == 0]
    .groupby('circuit_id')['race_delta']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'avg_delta', 'std': 'std_delta'})
    .sort_values('avg_delta', ascending=False)
)

fig, ax = plt.subplots(figsize=(13,9))

bars = ax.barh(
    delta_by_circuit['circuit_id'],
    delta_by_circuit['avg_delta'],
    color=bar_colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.75,
    xerr=delta_by_circuit['std_delta'] *0.3,
    error_kw={'ecolor': 'grey', 'capsize': 3, 'linewidth': 1}
)

for bar, val in zip(bars, delta_by_circuit['avg_delta']):
    ax.text(
        bar.get_width() + (0.1 if val >= 0 else -0.1),
        bar.get_y() + bar.get_height() /2,
        f'{val:+.2f}',
        va='center',
        ha='left' if val >= 0 else 'right',
        fontsize =9
    )

ax.axvline(0, color='black', linewidth =1.5, linestyle='-')
ax.set_xlabel('Average Position Gained (+) or Lost (-) vs Qualifying', fontsize=11)
ax.set_title(
    'Average Race Position Delta vs Grid Position by Circuit\n'
    '(DNFs excluded  |  Error bars = variability)',
    fontsize=13, fontweight='bold'
    )

ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_el = [
    Patch(facecolor='#27AE60', label='High overtaking (gain > 0.5)'),
    Patch(facecolor='#4A90D9', label='Neutral (-0.5 to +0.5)'),
    Patch(facecolor=F1_RED, label='Low overtaking (loss <-0.5)')
]

ax.legend(handles=legend_el, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('../output/04_race_delta_by_circuit.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:


#df.drop(columns=['race_delta'], inplace=True)

print()
print('KEY INSIGHT:')
high_overtake = delta_by_circuit.iloc[0]
low_overtake  = delta_by_circuit.iloc[-1]
print(f"  Highest overtaking circuit: {high_overtake['circuit_id']} "
      f"(avg +{high_overtake['avg_delta']:.2f} positions gained)")
print(f"  Lowest overtaking circuit:  {low_overtake['circuit_id']} "
      f"(avg {low_overtake['avg_delta']:.2f} positions)")
print()
print("  Low overtaking circuits → qualifying position is CRITICAL.")
print("  High overtaking circuits → race pace and strategy matter more.")
print("  This informs how the model will weight grid vs form features.")

# Summary

In [ ]:
engineered_features = [
    # Driver Form
    'driver_avg_finish_last_3_races',
    'driver_avg_points_last_3_races',
    'driver_avg_grid_last_3_races',
    'driver_points_cur_season',
    'driver_wins_cur_season',
    'driver_podiums_cur_season',
    'driver_dnf_rate',
    # Constructor Form
    'constructor_avg_finish_last_3_races',
    'constructor_avg_points_last_3_races',
    'constructor_points_current_season',
    'constructor_wins_current_season',
    'constructor_reliability',
    # Track Performance
    'driver_avg_finish_on_circuit',
    'driver_avg_points_on_circuit',
    'driver_best_finish_on_circuit',
    'driver_dnf_on_circuit',
    'driver_track_strength_score',
    # Teammate Comparison
    'driver_beats_teammate_rate',
    'driver_vs_teammate_finish_gap'
]

# Target variables
targets = ['position', 'points_finish', 'podium']

# Base features (from original dataset — these go into the model too)
base_features = ['grid']

# Combined feature set
all_features = engineered_features + base_features

# Verify all features exist
missing = [f for f in all_features if f not in df.columns]
if missing:
    print(f'WARNING: Missing columns: {missing}')
else:
    print(f'All {len(all_features)} features verified in dataset.')
    print(f'  Engineered features: {len(engineered_features)}')
    print(f'  Base features:       {len(base_features)}')
    print(f'  Target variables:    {len(targets)}')

# Check no NaN in features (should be clean from Notebook 03)
nan_check = df[all_features].isnull().sum().sum()
print(f'  NaN values in features: {nan_check}')


Feature-Target Correlations

In [ ]:
corr_with_targets = (
    df[all_features + targets]
    .corr()[targets]
    .loc[all_features]
    .sort_values('position', ascending=True)  # Sorted by position correlation
)

fig, ax = plt.subplots(figsize=(10, 12))

sns.heatmap(
    corr_with_targets,
    annot=True,
    fmt='.3f',
    cmap='RdYlGn_r',
    center=0,
    vmin=-0.7,
    vmax=0.7,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.6},
    ax=ax,
    annot_kws={'size': 10}
)

ax.set_title(
    'Feature Correlation with Target Variables\n'
    '(For position: negative = better predictor  |  '
    'For points_finish/podium: positive = better)',
    fontsize=13, fontweight='bold', pad=15
)
ax.set_xlabel('Target Variable', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('../output/04_feature_target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print('KEY INSIGHT:')
# Find strongest predictors for finish position
pos_corr = corr_with_targets['position'].abs().sort_values(ascending=False)
print('Top 5 strongest predictors for finish POSITION:')
for feat, val in pos_corr.head(5).items():
    direction = corr_with_targets.loc[feat, 'position']
    print(f'  {feat:<42} r = {direction:+.3f}')


Feature Strength Bar Chart

In [ ]:
# Use finish position as the primary target for ranking
feature_strength = (
    corr_with_targets['position']
    .abs()
    .sort_values(ascending=True)
    .reset_index()
)
feature_strength.columns = ['feature', 'abs_correlation']

# Color by feature group
group_colors = {}
for f in engineered_features[:7]:   # Driver form
    group_colors[f] = '#3498DB'
for f in engineered_features[7:12]: # Constructor form
    group_colors[f] = '#E67E22'
for f in engineered_features[12:17]:# Track performance
    group_colors[f] = '#27AE60'
for f in engineered_features[17:]:  # Teammate
    group_colors[f] = '#9B59B6'
group_colors['grid'] = F1_RED       # Base feature

bar_colors = [
    group_colors.get(f, '#888888')
    for f in feature_strength['feature']
]

fig, ax = plt.subplots(figsize=(12, 10))

bars = ax.barh(
    feature_strength['feature'],
    feature_strength['abs_correlation'],
    color=bar_colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.75
)

for bar, val in zip(bars, feature_strength['abs_correlation']):
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}',
        va='center', fontsize=9
    )

# Threshold lines
ax.axvline(0.40, color='black',  linestyle='--', linewidth=1,
           alpha=0.7, label='Strong signal  (|r| > 0.40)')
ax.axvline(0.25, color='grey',   linestyle='--', linewidth=1,
           alpha=0.7, label='Meaningful signal (|r| > 0.25)')
ax.axvline(0.10, color='#CCCCCC',linestyle='--', linewidth=1,
           alpha=0.7, label='Weak signal  (|r| < 0.10)')

ax.set_xlabel('Absolute Pearson Correlation with Finish Position', fontsize=12)
ax.set_title(
    'Feature Predictive Strength — Ranked by Correlation with Finish Position',
    fontsize=14, fontweight='bold'
)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498DB', label='Driver Form (7)'),
    Patch(facecolor='#E67E22', label='Constructor Form (5)'),
    Patch(facecolor='#27AE60', label='Track Performance (5)'),
    Patch(facecolor='#9B59B6', label='Teammate Comparison (2)'),
    Patch(facecolor=F1_RED,    label='Base Feature (grid)'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
ax.legend_.remove()

ax2 = ax.twinx()
ax2.set_yticks([])
line_legend = ax.legend(fontsize=8, loc='upper left')
ax.add_artist(line_legend)
feature_legend = ax.legend(handles=legend_elements, fontsize=8,
                            loc='lower right')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/04_feature_strength_ranked.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print('KEY INSIGHT:')
strongest = feature_strength.iloc[-1]
weakest   = feature_strength.iloc[0]
print(f"  Strongest predictor: {strongest['feature']} "
      f"(|r| = {strongest['abs_correlation']:.3f})")
print(f"  Weakest predictor:   {weakest['feature']} "
      f"(|r| = {weakest['abs_correlation']:.3f})")
weak_features = feature_strength[feature_strength.abs_correlation < 0.10]
if len(weak_features) > 0:
    print(f"\n  Features with |r| < 0.10 (candidates for review in Notebook 05):")
    for _, row in weak_features.iterrows():
        print(f"    {row['feature']} (|r| = {row['abs_correlation']:.3f})")


Feature-Feature Correlation

In [ ]:
feature_corr = df[all_features].corr()

fig, ax = plt.subplots(figsize=(18, 15))

# Mask upper triangle (matrix is symmetric — no need to show both halves)
mask = np.triu(np.ones_like(feature_corr, dtype=bool), k=1)

sns.heatmap(
    feature_corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.4,
    linecolor='white',
    cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.5},
    ax=ax,
    annot_kws={'size': 7}
)

ax.set_title(
    'Feature-Feature Correlation Matrix\n'
    '(Lower triangle only  |  |r| > 0.90 = potential multicollinearity)',
    fontsize=14, fontweight='bold', pad=15
)
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('../output/04_feature_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Flag highly correlated feature pairs
print('Feature pairs with |r| > 0.85 (multicollinearity candidates):')
print('-' * 65)
found_any = False
for i, f1 in enumerate(all_features):
    for j, f2 in enumerate(all_features):
        if j >= i:
            continue
        r = feature_corr.loc[f1, f2]
        if abs(r) > 0.85:
            print(f'  {f1:<42} ↔  {f2:<42}  r={r:+.3f}')
            found_any = True
if not found_any:
    print('  None found above 0.85 threshold.')


Scatter Analysis

In [ ]:
# Select top 4 features by absolute correlation with position
top4_features = (
    corr_with_targets['position']
    .abs()
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Sample for readability (scatter with 10k+ points gets noisy)
sample = df.sample(min(3000, len(df)), random_state=42)

for i, feat in enumerate(top4_features):
    ax = axes[i]

    r_val = corr_with_targets.loc[feat, 'position']

    ax.scatter(
        sample[feat],
        sample['position'],
        alpha=0.15,
        s=12,
        color='#3498DB',
        edgecolors='none'
    )

    # Trend line
    z = np.polyfit(sample[feat].dropna(), sample['position'].dropna(), 1)
    p = np.poly1d(z)
    x_line = np.linspace(sample[feat].min(), sample[feat].max(), 100)
    ax.plot(x_line, p(x_line), color=F1_RED, linewidth=2.5, label=f'r = {r_val:.3f}')

    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Finish Position', fontsize=9)
    ax.set_title(f'{feat}\n(r = {r_val:+.3f})', fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis()  # P1 at top (lower position = better)

plt.suptitle(
    'Top 4 Features vs Finish Position — Scatter Analysis\n'
    '(Y-axis inverted: P1 at top)',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('../output/04_feature_scatter_top4.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print('KEY INSIGHT:')
print("  Check for non-linear patterns — a curved relationship means")
print("  tree-based models (XGBoost, Random Forest) will outperform")
print("  linear models (Linear Regression) for this feature.")
print("  Any clear curve visible here is evidence for using Notebook 05's")
print("  advanced model selection over simple baselines.")


EDA Summary Statistics Table

In [ ]:

summary_table = pd.DataFrame({
    'feature': all_features,
    'corr_position':     [corr_with_targets.loc[f, 'position']     for f in all_features],
    'corr_points_finish':[corr_with_targets.loc[f, 'points_finish'] for f in all_features],
    'corr_podium':       [corr_with_targets.loc[f, 'podium']       for f in all_features],
    'mean':              [df[f].mean() for f in all_features],
    'std':               [df[f].std()  for f in all_features],
    'min':               [df[f].min()  for f in all_features],
    'max':               [df[f].max()  for f in all_features],
})

summary_table['abs_corr_position'] = summary_table['corr_position'].abs()
summary_table = summary_table.sort_values('abs_corr_position', ascending=False)
summary_table = summary_table.drop(columns='abs_corr_position')

print('FULL FEATURE SUMMARY TABLE')
print('=' * 100)
print(summary_table.round(3).to_string(index=False))
print()

# Save summary table
summary_table.to_csv('../output/04_feature_summary_table.csv', index=False)
print('Summary table saved: ../output/04_feature_summary_table.csv')
